In [1]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import color_palettes
importlib.reload(color_palettes)
from functions.color_palettes import *

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

c:\Users\aoezek\AppData\Local\miniconda3\envs\circadian\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# age bins/groups
bins_binary=[18,59,77]
labels_binary=["19-32","59-76"]

rename_dict = {
    "age" : "Age",
    "BMI": "Body Mass Index",
    "body_height": "Height (cm)",
    "body_weight": "Weight (kg)",
    "Resting_heart_rate": "Resting Heart Rate",
    "Systolic_BP": "Systolic Blood Pressure",
    "Diastolic_BP": "Diastolic Blood Pressure",
    
}

---

In [ ]:
participant_characteristics = pd.read_excel(metadata_path)
participant_characteristics.describe()

In [ ]:
participant_characteristics = clean_string_column(participant_characteristics,['completed_12days', 'completed_10days', 'complete_cgm_data', 'sex'])
print(participant_characteristics['completed_10days'].value_counts(),'\n')
print(participant_characteristics['completed_12days'].value_counts(),'\n')
print(participant_characteristics['complete_cgm_data'].value_counts())

In [ ]:
participant_characteristics = form_age_groups(
    participant_characteristics,
    age_col='age',
    new_col='age_group',
    bins=bins_binary,
    labels=labels_binary
)
participant_characteristics[['age', 'age_group','sex']]

In [7]:
participant_characteristics.to_excel(r"C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\metadata_clean.xlsx")
age_gender_map = participant_characteristics[['ID', 'age', 'age_group', 'sex']].copy()
age_gender_map['age_group_label'] = 'Age Group '+ age_gender_map['age_group'].astype(str)
age_gender_map['ID'] = age_gender_map['ID'].str.replace('AL', 'P', regex=False)
age_gender_map.to_excel(r'C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\excels\age_gender_map.xlsx')

In [ ]:
# for thesis text, dataset overview table (all participants)
results, groups = create_summary_table(
    participant_characteristics,
    group_col="age_group", 
    rename_dict=rename_dict
)
latex_table = to_latex(results, groups)
print(latex_table)

In [ ]:
participant_characteristics=participant_characteristics[(participant_characteristics['complete_cgm_data'] == 'yes') & 
                                                        (~participant_characteristics['ID'].isin(['AL28', 'AL43']))]

#28 and 43 has yes in that column but still not complete

In [ ]:
#for thesis text, dataset overview table valid participants (wrt to CGM data completeness)
results_final, groups_final = create_summary_table(
    participant_characteristics,
    group_col="age_group", 
    rename_dict=rename_dict
)

latex_table_final = to_latex(results_final, groups_final)
print(latex_table_final)

### Box Plots

In [ ]:
len(participant_characteristics['ID'].unique())

In [ ]:
save_dir="C:\\Users\\aoezek\\Desktop\\master_thesis\\master_thesis\\visualisations\\46_participants_meta\\"

In [ ]:
plot_boxplot_meta(participant_characteristics, 'BMI', save_dir)
plot_boxplot_meta(participant_characteristics, 'self_reported_sleep_duration', save_dir)
plot_boxplot_meta(participant_characteristics, '10day_glucose_avg', save_dir)
plot_boxplot_meta(participant_characteristics, 'Systolic_BP', save_dir)
plot_boxplot_meta(participant_characteristics, 'Diastolic_BP', save_dir)
plot_boxplot_meta(participant_characteristics, 'Resting_heart_rate', save_dir)

In [ ]:
analyze_difference(participant_characteristics, 'Systolic_BP', 'sex', {'age_group':'19-32'})
analyze_difference(participant_characteristics, 'Systolic_BP', 'age_group')

In [ ]:
# Grid plot 4 features
fig, axes = plt.subplots(2,2, figsize=(12,10))
for ax, var in zip(axes.flatten(), ['BMI', 'Systolic_BP', 'Diastolic_BP', 'Resting_heart_rate']):
    plot_boxplot_meta(participant_characteristics, var, ax=ax, save_dir=None)
plt.tight_layout()

os.makedirs(save_dir, exist_ok=True)
fig.savefig(
    os.path.join(save_dir, "Boxplot_Grid_significance_star_bothlevels.png"),
    dpi=400,
    bbox_inches="tight"
)
plt.show()